
# Métricas de evaluación y división de datos (train/test)
### Material introductorio para alumnos de Ingeniería Mecatrónica

---

## Objetivo
Comprender de manera sencilla por qué se dividen los datos en **entrenamiento** y **prueba**, y cómo evaluar el desempeño de un modelo de Machine Learning mediante métricas básicas de **clasificación** y **regresión**.

## ¿Qué aprenderás?
Al finalizar este notebook podrás:

- entender para qué sirve la división **train/test**,
- aplicar `train_test_split` en Python,
- identificar la diferencia entre evaluar clasificación y regresión,
- calcular métricas básicas de clasificación,
- calcular métricas básicas de regresión,
- interpretar resultados de manera sencilla.

---

## Contexto en Mecatrónica
En Ingeniería Mecatrónica, un modelo de Machine Learning puede utilizarse para tareas como:

- detectar fallas en motores,
- clasificar piezas aceptadas o defectuosas,
- predecir temperatura, corriente o velocidad,
- estimar variables de proceso a partir de sensores.

Pero no basta con entrenar un modelo. También es necesario saber:

1. **si realmente aprendió algo útil**, y  
2. **si funcionará bien con datos nuevos**.

Por eso usamos la **división de datos** y las **métricas de evaluación**.



## 1. ¿Por qué dividir los datos?

Cuando entrenamos un modelo con un conjunto de datos, el modelo aprende patrones a partir de esos ejemplos.

Si luego lo evaluamos usando exactamente los mismos datos con los que aprendió, podríamos obtener resultados engañosamente buenos.

### Idea básica
Por eso dividimos los datos en dos partes:

- **Train (entrenamiento):** datos que el modelo usa para aprender.
- **Test (prueba):** datos que el modelo no vio durante el entrenamiento y que se usan para evaluar su desempeño.

Esto permite responder una pregunta importante:

> **¿Qué tan bien funciona el modelo con datos nuevos?**



## 2. Esquema general

Una división muy común es:

- **80% para entrenamiento**
- **20% para prueba**

También se usan otras combinaciones, por ejemplo:

- 70% / 30%
- 75% / 25%

No existe una única regla absoluta, pero la idea siempre es la misma:
entrenar con una parte y evaluar con otra.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

pd.set_option("display.precision", 3)



## 3. Ejemplo de división de datos

Supongamos un problema de clasificación en un motor, usando:

- **temperatura**,
- **vibración**,

para determinar si el sistema está en:

- **0 = normal**
- **1 = posible falla**


In [ ]:

temperatura = np.array([28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45], dtype=float)
vibracion  = np.array([0.8, 0.9, 1.0, 1.0, 1.1, 1.2, 1.3, 1.5, 1.6, 1.8, 2.0, 2.1, 2.3, 2.5, 2.7, 2.9, 3.1, 3.3], dtype=float)
falla      = np.array([0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   1,   1,   1,   1,   1,   1,   1,   1], dtype=int)

df_clas = pd.DataFrame({
    "Temperatura_C": temperatura,
    "Vibracion_mm_s": vibracion,
    "Falla": falla
})

df_clas



## 4. Separar entradas y salida

En Machine Learning normalmente usamos:

- **X** para las variables de entrada,
- **y** para la variable objetivo o salida.


In [ ]:

X = df_clas[["Temperatura_C", "Vibracion_mm_s"]]
y = df_clas["Falla"]

X.head()



## 5. División con `train_test_split`

Usaremos `train_test_split` de `scikit-learn`.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_test :", X_test.shape)
print("Tamaño de y_train:", y_train.shape)
print("Tamaño de y_test :", y_test.shape)



### ¿Qué significa cada parámetro?
- `test_size=0.2`: reserva el 20% de los datos para prueba.
- `random_state=42`: fija la aleatoriedad para obtener siempre la misma división al ejecutar el código.



## 6. Entrenar con el conjunto de entrenamiento

Usaremos un modelo sencillo de **regresión logística** para la clasificación.


In [ ]:

modelo_clas = LogisticRegression()
modelo_clas.fit(X_train, y_train)

print("Modelo entrenado correctamente.")



## 7. Predecir sobre el conjunto de prueba

Aquí está la idea clave:

> El modelo se entrena con `train`, pero se evalúa con `test`.


In [ ]:

y_pred = modelo_clas.predict(X_test)

resultado_test = X_test.copy()
resultado_test["Real"] = y_test.values
resultado_test["Prediccion"] = y_pred
resultado_test



# Parte A. Métricas de clasificación

Cuando el problema consiste en predecir clases, por ejemplo:

- normal / falla,
- aceptada / defectuosa,
- alarma / no alarma,

se usan métricas de **clasificación**.



## 8. Accuracy

La **accuracy** es la proporción de predicciones correctas respecto al total.

\[
Accuracy = \frac{\text{predicciones correctas}}{\text{total de predicciones}}
\]

Es fácil de entender, pero no siempre es suficiente por sí sola.


In [ ]:

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")



## 9. Matriz de confusión

La matriz de confusión muestra cuántos casos se clasificaron correctamente e incorrectamente.


In [ ]:

matriz = confusion_matrix(y_test, y_pred)
print("Matriz de confusión:")
print(matriz)



### Interpretación de una matriz de confusión binaria

En problemas binarios, normalmente se interpreta así:

- **Verdaderos negativos (VN):** casos normales clasificados como normales.
- **Falsos positivos (FP):** casos normales clasificados como falla.
- **Falsos negativos (FN):** casos con falla clasificados como normales.
- **Verdaderos positivos (VP):** casos con falla clasificados como falla.

En aplicaciones reales, los **falsos negativos** pueden ser especialmente delicados si representan fallas que el sistema no detectó.



## 10. Precisión, recall y F1-score

Estas métricas ayudan a entender mejor el comportamiento del modelo.


In [ ]:

precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")



### Interpretación sencilla

- **Precisión (Precision):** de los casos que el modelo dijo que eran falla, ¿cuántos sí eran falla?
- **Recall:** de todas las fallas reales, ¿cuántas detectó el modelo?
- **F1-score:** combina precisión y recall en una sola medida.

### ¿Cuál es importante?
Depende del problema:

- Si quieres evitar falsas alarmas, te importa mucho la **precisión**.
- Si quieres no dejar pasar fallas reales, te importa mucho el **recall**.


In [ ]:

print(classification_report(y_test, y_pred, zero_division=0))



## 11. Ejemplo de interpretación en Mecatrónica

Supón que el modelo detecta fallas en un motor:

- Un **falso positivo** puede generar mantenimiento innecesario.
- Un **falso negativo** puede dejar pasar una falla real.

Por eso, en sistemas críticos, el **recall** puede ser más importante que solo la accuracy.



# Parte B. Métricas de regresión

Cuando el problema consiste en predecir un valor numérico continuo, por ejemplo:

- temperatura,
- corriente,
- velocidad,
- posición,

se usan métricas de **regresión**.



## 12. Ejemplo sencillo de regresión

Supongamos que queremos predecir la velocidad de un motor a partir del voltaje aplicado.


In [ ]:

voltaje = np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], dtype=float)
rpm = np.array([180, 270, 380, 500, 610, 700, 820, 910, 1010, 1100, 1210], dtype=float)

df_reg = pd.DataFrame({
    "Voltaje_V": voltaje,
    "Velocidad_RPM": rpm
})

df_reg



## 13. División train/test para regresión


In [ ]:

X_reg = df_reg[["Voltaje_V"]]
y_reg = df_reg["Velocidad_RPM"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg,
    test_size=0.25,
    random_state=42
)

print("Entrenamiento:", X_train_reg.shape)
print("Prueba:", X_test_reg.shape)



## 14. Entrenamiento de un modelo lineal


In [ ]:

modelo_reg = LinearRegression()
modelo_reg.fit(X_train_reg, y_train_reg)

print("Pendiente:", modelo_reg.coef_[0])
print("Intercepto:", modelo_reg.intercept_)


In [ ]:

y_pred_reg = modelo_reg.predict(X_test_reg)

resultado_reg = X_test_reg.copy()
resultado_reg["RPM_real"] = y_test_reg.values
resultado_reg["RPM_predicha"] = y_pred_reg
resultado_reg



## 15. Error absoluto medio (MAE)

El **MAE** mide, en promedio, cuánto se equivoca el modelo en unidades reales.

\[
MAE = \frac{1}{n} \sum |y_{real} - y_{predicho}|
\]

Si el resultado está en RPM, el error también queda en RPM.


In [ ]:

mae = mean_absolute_error(y_test_reg, y_pred_reg)
print(f"MAE: {mae:.4f}")



## 16. Error cuadrático medio (MSE)

El **MSE** penaliza más los errores grandes porque eleva al cuadrado las diferencias.

\[
MSE = \frac{1}{n} \sum (y_{real} - y_{predicho})^2
\]


In [ ]:

mse = mean_squared_error(y_test_reg, y_pred_reg)
print(f"MSE: {mse:.4f}")



## 17. Raíz del error cuadrático medio (RMSE)

El **RMSE** es la raíz cuadrada del MSE y vuelve a expresar el error en las mismas unidades de la variable original.


In [ ]:

rmse = np.sqrt(mse)
print(f"RMSE: {rmse:.4f}")



## 18. Coeficiente de determinación \(R^2\)

El coeficiente \(R^2\) indica qué tan bien explica el modelo la variación de los datos.

- Si está cerca de **1**, el ajuste es muy bueno.
- Si está cerca de **0**, el modelo explica poco.


In [ ]:

r2 = r2_score(y_test_reg, y_pred_reg)
print(f"R^2: {r2:.4f}")



## 19. Visualización del resultado de regresión


In [ ]:

plt.figure(figsize=(8,5))
plt.scatter(X_train_reg, y_train_reg, label="Entrenamiento")
plt.scatter(X_test_reg, y_test_reg, label="Prueba")
x_linea = np.linspace(X_reg.min().values[0], X_reg.max().values[0], 100).reshape(-1, 1)
y_linea = modelo_reg.predict(x_linea)
plt.plot(x_linea, y_linea, label="Modelo lineal")
plt.title("Regresión lineal con división train/test")
plt.xlabel("Voltaje (V)")
plt.ylabel("Velocidad (RPM)")
plt.grid(True)
plt.legend()
plt.show()



## 20. Interpretación sencilla de las métricas de regresión

- **MAE**: error promedio en unidades reales.
- **MSE**: penaliza más los errores grandes.
- **RMSE**: error típico en unidades reales.
- **R²**: qué tan bien explica el modelo los datos.

En Mecatrónica, si predices por ejemplo velocidad, corriente o temperatura, estas métricas permiten saber si el modelo es suficientemente preciso para el uso deseado.



## 21. ¿Por qué no basta con entrenar y medir sobre train?

Si un modelo se evalúa solo con los datos de entrenamiento, puede parecer excelente aunque en realidad no generalice bien.

Esto se relaciona con el problema de **sobreajuste** (*overfitting*), cuando el modelo aprende demasiado bien los ejemplos vistos pero falla con datos nuevos.

La división train/test ayuda a detectar este problema de forma básica.



## 22. Resumen general

### Para clasificación
Las métricas más comunes son:

- accuracy,
- matriz de confusión,
- precision,
- recall,
- F1-score.

### Para regresión
Las métricas más comunes son:

- MAE,
- MSE,
- RMSE,
- R².



## 23. Conclusión

Dividir los datos en **train** y **test** es una práctica básica y esencial en Machine Learning.

Además, usar métricas adecuadas permite evaluar si un modelo realmente es útil para un problema de Ingeniería Mecatrónica.

No todas las métricas significan lo mismo, y elegir cuál observar depende del tipo de problema:

- **clasificación** → métricas de clases,
- **regresión** → métricas de error numérico.



# Ejercicios propuestos

## Ejercicio 1. Comprensión conceptual
Responde con tus propias palabras:

1. ¿Para qué sirve dividir los datos en entrenamiento y prueba?
2. ¿Qué puede pasar si evaluamos el modelo con los mismos datos con los que se entrenó?
3. ¿Cuál es la diferencia entre un problema de clasificación y uno de regresión?

---

## Ejercicio 2. Clasificación de fallas
Crea un conjunto de datos con:

- temperatura,
- vibración,
- clase `0 = normal`, `1 = falla`.

Después:

1. divide los datos con `train_test_split`,
2. entrena un modelo de clasificación,
3. calcula accuracy, precision, recall y F1,
4. interpreta qué métrica te parece más importante para este caso.

---

## Ejercicio 3. Predicción de velocidad
Usa datos de ejemplo con:

- voltaje,
- velocidad del motor.

Después:

1. divide los datos en train/test,
2. entrena un modelo lineal,
3. calcula MAE, MSE, RMSE y R²,
4. interpreta el significado físico del error obtenido.

---

## Ejercicio 4. Matriz de confusión
Con un problema de clasificación binaria:

1. obtén la matriz de confusión,
2. identifica VN, FP, FN y VP,
3. explica cuál tipo de error sería más crítico en una aplicación mecatrónica real.

---

## Ejercicio 5. Reflexión aplicada
Menciona un caso real de Ingeniería Mecatrónica donde:

- usarías clasificación,
- usarías regresión,

y explica qué métricas evaluarías en cada uno.



# Actividad integradora sugerida

## Mini proyecto
Plantea dos pequeños problemas de Machine Learning relacionados con mecatrónica:

### Problema 1: Clasificación
Ejemplo:
- falla / no falla,
- pieza aceptada / defectuosa.

### Problema 2: Regresión
Ejemplo:
- voltaje vs velocidad,
- corriente vs carga.

Entrega:

1. descripción de ambos problemas,
2. tabla de datos,
3. división train/test,
4. entrenamiento de un modelo para cada caso,
5. métricas de evaluación correspondientes,
6. interpretación técnica de los resultados,
7. conclusión breve.
